# Notebook 05: Production Pipeline and Integration (Bonus)

Merge the adapter, build an inference pipeline, and integrate the fine-tuned model with the agentic RAG system from Part 5.

## 1. Setup

In [ ]:
import os
import json
import time
import random
import re
import torch
from pathlib import Path
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'
ADAPTER_PATH = Path('../results/ecommerce-ft-v1/adapter')

print(f'[OK] Setup complete')
print(f'  Model: {MODEL_ID}')
print(f'  Adapter: {ADAPTER_PATH}')

## 2. Merge Adapter into Base Model

During training, we keep the adapter separate from the base model. For production, we **merge** them into a single model for faster inference (no adapter overhead).

In [ ]:
# Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=True,
    attn_implementation="eager"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load adapter
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))

print(f'[OK] Model + adapter loaded')

In [ ]:
# Merge adapter weights into base model
model = model.merge_and_unload()

print(f'[OK] Adapter merged into base model')
print(f'  The model is now a single merged model (no adapter overhead)')

In [ ]:
# Save the merged model
merged_path = Path('../results/ecommerce-ft-merged')
merged_path.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(merged_path))
tokenizer.save_pretrained(str(merged_path))

# Check sizes
adapter_size = sum(f.stat().st_size for f in ADAPTER_PATH.rglob('*') if f.is_file()) / (1024**2)
merged_size = sum(f.stat().st_size for f in merged_path.rglob('*') if f.is_file()) / (1024**2)

print(f'[OK] Merged model saved to {merged_path}')
print(f'  Adapter size: {adapter_size:.0f} MB')
print(f'  Merged model size: {merged_size:.0f} MB')

## 3. Build Inference Pipeline

In [ ]:
def generate_response(query, system_prompt=None, max_new_tokens=256):
    """Generate an e-commerce customer service response."""
    model.eval()
    
    if system_prompt is None:
        system_prompt = 'You are a helpful and professional customer service agent for ShopEasy.'
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': query}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True, return_dict=True
    ).to(model.device)
    
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id
            )
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    return response.strip()

# Quick test
test = generate_response('What is your return policy?')
print(f'Test: {test}')
print(f'\n[OK] Inference pipeline ready')

In [ ]:
# Benchmark inference speed
test_queries = [
    'What is your return policy?',
    'How long does shipping take?',
    'Do you offer international shipping?',
    'Can I cancel my order?',
    'What payment methods do you accept?'
]

times = []
total_tokens = 0

for query in test_queries:
    start = time.time()
    response = generate_response(query)
    elapsed = time.time() - start
    
    output_tokens = len(tokenizer.encode(response))
    total_tokens += output_tokens
    times.append(elapsed)

avg_time = sum(times) / len(times)
tokens_per_sec = total_tokens / sum(times)

print(f'Inference Benchmark ({len(test_queries)} queries):')
print(f'  Avg latency: {avg_time:.2f}s per query')
print(f'  Throughput: {tokens_per_sec:.1f} tokens/sec')
print(f'  Total tokens: {total_tokens}')

## 4. Quantized Inference Options

For deploying beyond a single GPU, there are several options:

| Format | Use Case | Pros | Cons |
|--------|----------|------|------|
| **HuggingFace** (current) | Development, single GPU | Easy, flexible | Requires GPU |
| **GGUF** (llama.cpp) | CPU/edge deployment | Runs on CPU, mobile | Slower inference |
| **vLLM** | Production serving | High throughput, batching | More setup |
| **TGI** (HF) | Production serving | Docker-ready, streaming | More setup |

For this bootcamp, we stay with HuggingFace format. In production, you'd typically:
1. Export to GGUF for edge/CPU deployment
2. Use vLLM or TGI for GPU serving at scale

## 5. Integration with Agentic RAG (Part 5)

This is the big payoff: we replace the API-based LLM in our Part 5 agent with our **local fine-tuned model**.

The agent architecture stays the same:
```
User Query → Agent Loop → Tools → Response
```

But instead of calling an external API for the LLM brain, we use our local model.

In [ ]:
# ============================================================
# Re-create the tool registry from Part 5
# ============================================================

MOCK_ORDERS = {
    'ORD-12345': {'status': 'shipped', 'carrier': 'FedEx', 'estimated_delivery': '2026-02-03',
                  'items': ['Blue iPhone 15 Case'], 'destination': 'New York, NY'},
    'ORD-67890': {'status': 'processing', 'estimated_delivery': '2026-02-05',
                  'items': ['Wireless Earbuds'], 'destination': 'Miami, FL'}
}

MOCK_WEATHER = {
    'miami': {'condition': 'Hurricane Warning', 'delay_days': 3},
    'new york': {'condition': 'Clear', 'delay_days': 0}
}

MOCK_INVENTORY = {
    'iphone 15 case': {'blue': {'in_stock': True, 'quantity': 42}, 'black': {'in_stock': False}},
    'airpods pro': {'default': {'in_stock': True, 'quantity': 120}}
}

# Tool functions (same as Part 5)
def check_order_status(order_id: str) -> str:
    order_id = order_id.upper().strip()
    if not order_id.startswith('ORD-'):
        order_id = f'ORD-{order_id}'
    if order_id in MOCK_ORDERS:
        return json.dumps({'success': True, 'order_id': order_id, **MOCK_ORDERS[order_id]})
    return json.dumps({'success': False, 'error': f'Order {order_id} not found'})

def get_weather_alerts(location: str) -> str:
    key = location.lower().split(',')[0].strip()
    if key in MOCK_WEATHER:
        return json.dumps({'success': True, 'location': location, **MOCK_WEATHER[key]})
    return json.dumps({'success': True, 'location': location, 'condition': 'Clear', 'delay_days': 0})

def check_inventory(product_name: str) -> str:
    key = product_name.lower().strip()
    for pkey in MOCK_INVENTORY:
        if pkey in key or key in pkey:
            return json.dumps({'success': True, 'product': pkey.title(), 'variants': MOCK_INVENTORY[pkey]})
    return json.dumps({'success': False, 'error': 'Product not found'})

def create_return_request(order_id: str, reason: str) -> str:
    order_id = order_id.upper().strip()
    if not order_id.startswith('ORD-'):
        order_id = f'ORD-{order_id}'
    if order_id not in MOCK_ORDERS:
        return json.dumps({'success': False, 'error': 'Order not found'})
    return_id = f'RET-{random.randint(10000, 99999)}'
    return json.dumps({'success': True, 'return_id': return_id, 'order_id': order_id})

# Tool registry
TOOLS = {
    'check_order_status': check_order_status,
    'get_weather_alerts': get_weather_alerts,
    'check_inventory': check_inventory,
    'create_return_request': create_return_request,
}

TOOL_DESCRIPTIONS = """Available tools:
- check_order_status(order_id): Check order status and tracking
- get_weather_alerts(location): Check weather alerts for shipping delays
- check_inventory(product_name): Check product availability
- create_return_request(order_id, reason): Process a return

To use a tool, respond with:
TOOL_CALL: tool_name(arg1, arg2)

After receiving tool results, provide the final answer to the customer."""

print(f'[OK] {len(TOOLS)} tools registered')

In [ ]:
def local_agent(query, max_iterations=3):
    """ReAct agent using the local fine-tuned model."""
    model.eval()
    
    system_prompt = f"""You are a helpful e-commerce customer service agent for ShopEasy.

{TOOL_DESCRIPTIONS}

If the customer's question can be answered from your training, answer directly.
If you need real-time data (order status, weather, inventory), use a tool."""
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': query}
    ]
    
    tools_used = []
    
    for iteration in range(max_iterations):
        # Generate response
        inputs = tokenizer.apply_chat_template(
            messages, return_tensors='pt', add_generation_prompt=True, return_dict=True
        ).to(model.device)
        
        with torch.no_grad():
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs = model.generate(
                    **inputs, max_new_tokens=256,
                    temperature=0.3, do_sample=True, pad_token_id=tokenizer.pad_token_id
                )
        
        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()
        
        # Check for tool calls
        tool_match = re.search(r'TOOL_CALL:\s*(\w+)\((.*)\)', response)
        
        if tool_match:
            tool_name = tool_match.group(1)
            tool_args_raw = tool_match.group(2)
            
            if tool_name in TOOLS:
                # Parse arguments
                args = [a.strip().strip('"').strip("'") for a in tool_args_raw.split(',')]
                
                # Execute tool
                try:
                    result = TOOLS[tool_name](*args)
                    tools_used.append(tool_name)
                except Exception as e:
                    result = json.dumps({'error': str(e)})
                
                # Add tool interaction to messages
                messages.append({'role': 'assistant', 'content': response})
                messages.append({'role': 'user', 'content': f'Tool result: {result}'})
            else:
                break  # Unknown tool, return what we have
        else:
            break  # No tool call, return the response
    
    return {
        'response': response,
        'tools_used': tools_used
    }

print('[OK] Local agent ready')

In [ ]:
# Test the local agent with the same scenarios from Part 5
test_scenarios = [
    'What is your return policy?',                                          # Direct answer (no tool)
    'What is the status of order ORD-12345?',                              # Tool: check_order_status
    'Will my package to Miami be delayed due to weather?',                  # Tool: get_weather_alerts
    'Is the blue iPhone 15 case in stock?',                                # Tool: check_inventory
    'I want to return order ORD-67890 because it arrived damaged.',         # Tool: create_return_request
]

print('LOCAL FINE-TUNED AGENT')
print('=' * 70)

for query in test_scenarios:
    result = local_agent(query)
    tools = result['tools_used']
    
    print(f'\nQ: {query}')
    if tools:
        print(f'  Tools: {tools}')
    print(f'  A: {result["response"][:200]}')
    print('-' * 70)

## 6. Full Stack Architecture

```
+------------------------------------------------------------------+
|                        User Query                                 |
+------------------------------------------------------------------+
                              |
                              v
+------------------------------------------------------------------+
|                    FINE-TUNED LLM (Part 6)                        |
|                                                                    |
|  - Local model (Phi-3-mini + QLoRA adapter, merged)              |
|  - Trained on e-commerce FAQ + synthetic conversations           |
|  - Knows policies, tone, format without system prompt             |
+------------------------------------------------------------------+
                              |
                              v
+------------------------------------------------------------------+
|                    AGENT LOOP (Part 5)                            |
|                                                                    |
|  +---> REASON: What tool to use?                                  |
|  |     ACT: Execute tool                                          |
|  |     OBSERVE: Process results                                   |
|  +---- LOOP until solved                                          |
+------------------------------------------------------------------+
                              |
                              v
+------------------------------------------------------------------+
|                    TOOLS (Part 5)                                  |
|                                                                    |
|  [check_order_status]  [get_weather_alerts]  [check_inventory]   |
|  [create_return_request]  [search_knowledge_base]                |
+------------------------------------------------------------------+
                              |
                              v
+------------------------------------------------------------------+
|                RAG KNOWLEDGE BASE (Part 4)                        |
|                                                                    |
|  AWS Bedrock + OpenSearch (vector embeddings of FAQ + policies)   |
+------------------------------------------------------------------+
                              |
                              v
+------------------------------------------------------------------+
|                       Response                                    |
+------------------------------------------------------------------+
```

## 7. Cost and Performance Comparison

In [ ]:
# Build comparison table
comparison = {
    'Aspect': [
        'Latency',
        'Cost',
        'Privacy',
        'Customization',
        'Brand Voice',
        'Knowledge Cutoff',
        'Real-time Data',
        'Scalability'
    ],
    'API-based (Parts 4-5)': [
        '~1-3s (network round trip)',
        'Per-token pricing ($)',
        'Data sent to external API',
        'Prompt engineering only',
        'System prompt dependent',
        'Provider-determined',
        'Via tools (same)',
        'Provider handles scaling'
    ],
    'Fine-Tuned Local (Part 6)': [
        f'~{avg_time:.1f}s (local GPU)',
        'One-time GPU cost',
        'Data stays on your hardware',
        'Model weights trained on domain',
        'Learned from training data',
        'You control the training data',
        'Via tools (same)',
        'Need own GPU infrastructure'
    ]
}

import pandas as pd

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

## 8. Summary and Bootcamp Recap

### What We Built Across Parts 4-6

| Part | What | Key Skill |
|------|------|-----------|
| **Part 4** | RAG pipeline with vector search | Retrieval-Augmented Generation |
| **Part 5** | Agentic RAG with tools and reasoning | ReAct pattern, LangGraph |
| **Part 6** | Fine-tuned domain-specific LLM | QLoRA, SFT, DPO, evaluation |

### Part 6 Journey

```
NB01: Set up GPU, loaded Phi-3-mini model, saw generic baseline responses
NB02: Prepared training data from Part 4 FAQ + synthetic conversations, fine-tuned with QLoRA
NB03: Evaluated with ROUGE, semantic similarity, LLM-as-judge
NB04: (Bonus) Tuned hyperparameters, applied DPO for better tone
NB05: (Bonus) Merged adapter, built pipeline, integrated with Part 5 agent
```

### The Full Stack

You now have all the building blocks for a production AI assistant:
1. **Knowledge retrieval** (RAG) for accurate, up-to-date information
2. **Tool use** (Agents) for real-time actions and data
3. **Custom behavior** (Fine-tuning) for domain expertise and brand voice

These three capabilities are complementary -- the best AI systems combine all of them.

In [ ]:
print('=' * 60)
print('[OK] Part 6 Complete!')
print()
print('You now have a fine-tuned e-commerce LLM that:')
print('  - Knows your policies and FAQ (trained on Part 4 data)')
print('  - Speaks with your brand voice (SFT + DPO)')
print('  - Can use tools for real-time data (integrated with Part 5)')
print('  - Runs locally on your GPU (no API costs)')
print('=' * 60)